# Phase 3 Lab — Reference Solution

**Phase:** Data Visualization  
**Scenario:** An executive needs to understand where sales growth and profit are coming from and where margin risk is emerging.

**Deliverable:** A six-chart visual narrative with a one-page executive summary and accessibility notes.

Use this only after completing your own attempt. Compare design decisions—not merely output.

## Requirements

        1. Write the decision question for each chart before plotting.
2. Create trend, category comparison, distribution, and relationship views.
3. Annotate the most decision-relevant observation.
4. Show data coverage and sample size.
5. Avoid dual axes and unjustified truncated baselines.
6. Write one evidence-limitation-action statement per visual.
7. Review labels, legibility, and accessibility.

        ## Acceptance criteria

        - The notebook runs from a clean kernel in order.
        - Inputs and outputs have explicit contracts.
        - Invalid, missing, extreme, duplicate, and unseen cases are considered.
        - Important invariants use assertions or tests.
        - Results include interpretation and limitations.
        - Generated artifacts are written under the course `artifacts/` folder.

## Planning worksheet

Complete before coding:

| Question | Your answer |
|---|---|
| What decision or user does the result serve? | |
| What does one row/object/event represent? | |
| What are the required inputs and types? | |
| What outputs and side effects are allowed? | |
| Which assumptions are most fragile? | |
| What is the simplest valid baseline? | |
| Which edge cases must be tested? | |
| How will you know the result is correct? | |

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

_candidates = [Path.cwd(), *Path.cwd().parents]
COURSE_ROOT = next((p for p in _candidates if (p / "datasets").exists()), Path.cwd())
DATA_DIR = COURSE_ROOT / "datasets"
ARTIFACT_DIR = COURSE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(COURSE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Course root: {COURSE_ROOT}")

## Reference implementation

This is one defensible solution, not the only correct design. Identify at least one improvement before adopting it.

In [ ]:
sales=pd.read_csv(DATA_DIR/"retail_sales.csv",parse_dates=["order_date"])
monthly=sales.set_index("order_date").resample("MS").agg(revenue=("revenue","sum"),profit=("profit","sum"))
category=sales.groupby("category").agg(revenue=("revenue","sum"),profit=("profit","sum"))
category["margin"]=category["profit"]/category["revenue"]

fig,ax=plt.subplots(figsize=(9,4))
ax.plot(monthly.index,monthly["revenue"],marker="o")
peak=monthly["revenue"].idxmax()
ax.annotate(f"Peak: {monthly.loc[peak,'revenue']:,.0f}",xy=(peak,monthly.loc[peak,"revenue"]),
            xytext=(peak,monthly["revenue"].max()*1.08),arrowprops={"arrowstyle":"->"})
ax.set(title=f"Monthly revenue ({sales.order_date.min().date()} to {sales.order_date.max().date()})",
       xlabel="Month",ylabel="Revenue")
fig.autofmt_xdate(); plt.show()

ranked=category["profit"].sort_values()
fig,ax=plt.subplots(figsize=(7,4))
ax.barh(ranked.index,ranked.values)
ax.set(title=f"Profit by category, n={len(sales):,} orders",xlabel="Profit",ylabel="Category")
plt.show()

fig,ax=plt.subplots(figsize=(7,4))
ax.scatter(sales["discount_pct"],sales["profit"],alpha=.3)
ax.set(title="Discount and order profit",xlabel="Discount",ylabel="Profit")
plt.show()

display(category.sort_values("profit",ascending=False).round(3))
print("Evidence: profit concentration differs across categories.")
print("Limitation: synthetic observational data does not establish discount causality.")
print("Action: investigate category-specific discount experiments and contribution margin.")

## Solution review

Review the reference under four lenses:

1. **Correctness:** Are contracts and calculations enforced?
2. **Robustness:** What failures remain unhandled?
3. **Maintainability:** Which responsibilities should become modules/functions?
4. **Decision validity:** Do outputs support the stated use without overclaiming?

Extend the implementation with one additional test and one observability improvement.